In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 디바이스:", device)


In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame   # 피처 + 타깃이 합쳐진 DataFrame

print(f"데이터 크기: {df.shape}")          # (20640, 9)
print(f"\n피처 설명:")
for name, desc in zip(housing.feature_names, housing.DESCR.split('\n')[:30]):
    pass
df.head()


In [ ]:
# 기초 통계 및 타깃 분포 시각화
print(df.describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["MedHouseVal"], bins=50, color="steelblue", edgecolor="white")
axes[0].set_title("주택 가격 분포 (타깃)")
axes[0].set_xlabel("중위 주택 가격 ($10만)")
axes[0].set_ylabel("빈도")
axes[0].grid(alpha=0.3)

axes[1].scatter(df["MedInc"], df["MedHouseVal"],
                alpha=0.1, s=5, color="tomato")
axes[1].set_title("소득 수준 vs. 주택 가격")
axes[1].set_xlabel("중위 소득")
axes[1].set_ylabel("중위 주택 가격")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("housing_eda.png", dpi=100, bbox_inches="tight")
plt.show()


In [ ]:
X = housing.data.values.astype("float32")   # (20640, 8)
y = housing.target.values.astype("float32") # (20640,)

# 훈련 / 검증 / 테스트 분리 (6 : 2 : 2)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=42  # 0.25 × 0.8 = 0.2
)

print(f"훈련: {X_train.shape} | 검증: {X_val.shape} | 테스트: {X_test.shape}")

# 피처 정규화 — 훈련 세트 통계만 사용 (데이터 누수 방지)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype("float32")
X_val   = scaler.transform(X_val).astype("float32")
X_test  = scaler.transform(X_test).astype("float32")

print(f"\n스케일링 후 훈련 피처 평균: {X_train.mean(axis=0).round(3)}")
print(f"스케일링 후 훈련 피처 표준편차: {X_train.std(axis=0).round(3)}")


In [ ]:
class HousingDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE   = 256
train_loader = DataLoader(HousingDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(HousingDataset(X_val,   y_val),   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(HousingDataset(X_test,  y_test),  batch_size=BATCH_SIZE, shuffle=False)

print(f"훈련 {len(X_train):,}개 ({len(train_loader)} 배치) | "
      f"검증 {len(X_val):,}개 | 테스트 {len(X_test):,}개")


In [ ]:
# 8 → 128 → 64 → 32 → 1 (선형 회귀 출력)
class HousingNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(8, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)   # 활성화 함수 없음 — 연속값 출력
        )

    def forward(self, x):
        return self.net(x).squeeze(1)   # (배치, 1) → (배치,)

model = HousingNet().to(device)
print(model)
print(f"전체 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

# 추론 테스트
dummy = torch.randn(4, 8).to(device)
print(f"\n입력 shape:  {dummy.shape}")          # (4, 8)
print(f"출력 shape:  {model(dummy).shape}")     # (4,)
print(f"출력값 예시: {model(dummy).detach()}")   # 연속값 (음수 포함 가능)


In [ ]:
EPOCHS = 100
LR     = 0.001

# 회귀 손실함수: MSELoss (잔차 제곱 평균)
# HuberLoss는 이상치에 덜 민감한 대안 — delta 파라미터로 MSE/MAE 전환점 조정
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=10
)

history    = {"train_loss": [], "val_loss": [], "val_rmse": []}
best_rmse  = float("inf")

print(f"{'Epoch':>6} | {'Train MSE':>10} | {'Val MSE':>10} | {'Val RMSE':>9}")
print("-" * 45)

for epoch in range(1, EPOCHS + 1):
    # ── 훈련 ──────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        pred = model(X_b)
        loss = criterion(pred, y_b)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # ── 검증 ──────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            pred      = model(X_b)
            val_loss += criterion(pred, y_b).item()
            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(y_b.cpu().numpy())
    val_loss /= len(val_loader)
    val_rmse  = mean_squared_error(all_labels, all_preds)
    # val_rmse  = root_mean_squared_error(all_labels, all_preds)

    scheduler.step(val_loss)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_rmse"].append(val_rmse)

    if val_rmse < best_rmse:
        best_rmse = val_rmse
        torch.save(model.state_dict(), "housing_best.pt")

    if epoch % 20 == 0:
        print(f"{epoch:>6} | {train_loss:>10.4f} | {val_loss:>10.4f} | {val_rmse:>8.4f}")

print(f"\n최고 검증 RMSE: {best_rmse:.4f} → housing_best.pt 저장")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["train_loss"], label="훈련 MSE")
axes[0].plot(history["val_loss"],   label="검증 MSE")
axes[0].set_title("손실 곡선 (MSE)")
axes[0].set_xlabel("에폭")
axes[0].set_ylabel("MSE")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history["val_rmse"], color="darkorange")
axes[1].axhline(y=0.5, color="red", linestyle="--", alpha=0.6, label="목표 RMSE 0.50")
axes[1].set_title("검증 RMSE 곡선")
axes[1].set_xlabel("에폭")
axes[1].set_ylabel("RMSE ($10만)")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("housing_training_curve.png", dpi=100, bbox_inches="tight")
plt.show()


In [ ]:
# 최고 모델 불러오기
model.load_state_dict(torch.load("housing_best.pt", map_location=device))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for X_b, y_b in test_loader:
        pred = model(X_b.to(device))
        all_preds.extend(pred.cpu().numpy())
        all_labels.extend(y_b.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

mae  = mean_absolute_error(all_labels, all_preds)
rmse = mean_squared_error(all_labels, all_preds)
r2   = r2_score(all_labels, all_preds)

print("=== 테스트 세트 평가 결과 ===")
print(f"  MAE  : {mae:.4f}  (평균 절대 오차, $10만 단위 → 약 ${mae*10:.1f}만)")
print(f"  RMSE : {rmse:.4f}  (이상치에 민감한 오차)")
print(f"  R²   : {r2:.4f}  (1.0에 가까울수록 좋음)")


In [ ]:
import pickle

# 스케일러도 함께 저장 (추론 시 동일한 변환 필요)
with open("housing_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

torch.save({
    "model_state_dict":     model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "best_rmse": best_rmse,
    "history":   history,
}, "housing_checkpoint.pt")

print("저장 완료:")
print("  - housing_best.pt        (추론용 — state_dict만)")
print("  - housing_checkpoint.pt  (재훈련용 — 전체 상태)")
print("  - housing_scaler.pkl     (추론 시 피처 스케일링용)")
